# MG-1 and MG-2 on a CPU

Execute from the first cell in a fresh kernel. This notebook runs the actual forward path, all 600 training updates, checkpoint continuation, a fresh-process repeat and generation. Outputs go to a temporary directory so the reference experiment remains unchanged. The curve and all original failure probes are kept in the repository, and the chapter presents the decisive evidence statically.


In [1]:
from pathlib import Path
import json
import sys
import numpy as np
ROOT = Path.cwd()
assert (ROOT / "src/config/book.mjs").is_file(), "Run from the book repository root"
sys.path.insert(0, str(ROOT / "code/part-iv"))
sys.path.insert(0, str(ROOT / "code/mini-gpt"))
import runpy
import tempfile
forward_module = runpy.run_path(str(ROOT / "code/mini-gpt/forward.py"))
forward = forward_module["run"]()
assert forward["parameter_count"] == 13875 and forward["scored_events"] == 32
np.testing.assert_allclose(forward["loss"], forward["scalar_mean_loss"], atol=1e-12, rtol=0)
np.testing.assert_allclose(forward["zero_logits_loss"], np.log(291), atol=1e-12, rtol=0)
print("MG-1", {key: forward[key] for key in ("parameter_count", "scored_events", "loss", "zero_logits_loss")})
print("shapes", forward["shapes"])


MG-1 {'parameter_count': 13875, 'scored_events': 32, 'loss': 5.670998865509144, 'zero_logits_loss': 5.673323267171493}
shapes {'token_embedding': [2, 64, 16], 'position_embedding': [64, 16], 'block_input': [2, 64, 16], 'norm1': [2, 64, 16], 'q_projection': [2, 64, 16], 'k_projection': [2, 64, 16], 'v_projection': [2, 64, 16], 'q_heads': [2, 2, 64, 8], 'k_heads': [2, 2, 64, 8], 'v_heads': [2, 2, 64, 8], 'scores': [2, 2, 64, 64], 'visible': [2, 1, 64, 64], 'weights': [2, 2, 64, 64], 'mixtures': [2, 2, 64, 8], 'joined': [2, 64, 16], 'attention_output': [2, 64, 16], 'residual1': [2, 64, 16], 'norm2': [2, 64, 16], 'ffn_linear': [2, 64, 64], 'ffn_relu': [2, 64, 64], 'ffn_output': [2, 64, 16], 'block_output': [2, 64, 16], 'final_norm': [2, 64, 16], 'logits': [2, 64, 291]}


## Train, resume and repeat

The pipeline verifies all continuation steps and final model/optimizer/RNG states. Its fresh process repeats from initialization. Runtime is a local measurement, not a performance promise. The saved curve is a comparison record, while the scalar/causal/loss tests independently validate mathematics.


In [2]:
training_module = runpy.run_path(str(ROOT / "code/mini-gpt/run.py"))
reference_run = json.loads((ROOT / "data/mini-gpt/training-run.json").read_text())
with tempfile.TemporaryDirectory(prefix="book-mg-notebook-") as temporary:
    destination = Path(temporary)
    training_module["pipeline"](destination)
    measured = json.loads((destination / "training-run.json").read_text())
    generations = json.loads((destination / "generation.json").read_text())
    assert all(measured["resumed_checks"].values())
    assert all(measured["fresh_process_checks"].values())
    assert measured["selected_step"] == 50
    for actual, expected in zip(measured["curve"], reference_run["curve"], strict=True):
        assert actual["step"] == expected["step"]
        np.testing.assert_allclose([actual["train_loss"], actual["validation_loss"]], [expected["train_loss"], expected["validation_loss"]], atol=1e-9, rtol=0)
print("measured environment", measured["environment"])
print("measured total seconds", measured["total_elapsed_seconds"])


{
  "selected_step": 50,
  "curve": [
    {
      "step": 0,
      "train_loss": 5.674426629050179,
      "validation_loss": 5.669803297874319
    },
    {
      "step": 50,
      "train_loss": 1.2694209776589775,
      "validation_loss": 2.405571231866604
    },
    {
      "step": 100,
      "train_loss": 0.36875937932816544,
      "validation_loss": 2.70305284990864
    },
    {
      "step": 150,
      "train_loss": 0.2163515724970035,
      "validation_loss": 2.993821681869046
    },
    {
      "step": 200,
      "train_loss": 0.17665242747723808,
      "validation_loss": 3.3247100575925836
    },
    {
      "step": 250,
      "train_loss": 0.1754224894809983,
      "validation_loss": 3.5228849937366404
    },
    {
      "step": 300,
      "train_loss": 0.1680241568647308,
      "validation_loss": 3.5958033289935996
    },
    {
      "step": 350,
      "train_loss": 0.16676187952231675,
      "validation_loss": 3.748807172738595
    },
    {
      "step": 400,
      "train_los

## Inspect success and failure together

The final snapshot can reproduce a familiar aligned red suffix and still answer red after an explicit change to blue. The earlier selected snapshot has lower development loss without succeeding on every prompt. These are diagnostic examples, not a general capability benchmark.


In [3]:
for entry in generations:
    if entry["mode"] == "greedy":
        print(entry["checkpoint"], repr(entry["prompt"]), "=>", repr(entry["completion"]), "stop", entry["stop"])
final_red = next(entry for entry in generations if entry["checkpoint"] == "final" and entry.get("probe_id") == "aligned-red" and entry["mode"] == "greedy")
assert final_red["completion"] == "red."
print("complete red trace", final_red["trace"])
assert measured["curve"][-1]["train_loss"] < measured["curve"][1]["train_loss"]
assert measured["curve"][-1]["validation_loss"] > measured["curve"][1]["validation_loss"]
print("MG-2 evidence complete within the recorded stack; cross-stack equality is not claimed.")


selected 'The key is red. Answer:' => ' of Anspitap the blue.' stop EOS
selected 'The key is blue. Answer:' => 'door.' stop EOS
selected 'The key was red. Now blue. Answer:' => 'pped.' stop EOS
selected 'Old limit: 750. New limit:' => ': sus.' stop EOS
selected 'The key is red. Answer: ' => 'bluesuersther: blue key.' stop EOS
selected 'The key is blue. Answer: ' => 'blue.' stop EOS
selected 'The key was red. Now blue. Answer: ' => 'blue key.' stop EOS
selected 'Old limit: 600. New limit: 750. Answer: ' => 'peng.' stop EOS
final 'The key is red. Answer:' => 'Result: keyred red.' stop EOS
final 'The key is blue. Answer:' => 'blue.' stop EOS
final 'The key was red. Now blue. Answer:' => 'wersionew.' stop EOS
final 'Old limit: 750. New limit:' => ': Paris.' stop EOS
final 'The key is red. Answer: ' => 'red.' stop EOS
final 'The key is blue. Answer: ' => 'blue.' stop EOS
final 'The key was red. Now blue. Answer: ' => 'red.' stop EOS
final 'Old limit: 600. New limit: 750. Answer: ' => 'shipe